# LLaDA 8B LoRA Fine-Tuning — Voice

Fine-tune LLaDA 8B (a **masked diffusion** model) on essay pairs to capture writing voice.
Uses QLoRA so it fits on a single GPU.

**How LLaDA differs from Llama/GPT-2:**
- Autoregressive models generate left-to-right, one token at a time.
- LLaDA starts with a fully masked sequence and iteratively unmasks tokens — more like editing than typing.
- Training: randomly mask response tokens, predict them. Loss = cross-entropy / masking probability.
- Inference: start fully masked → predict all → remask lowest-confidence → repeat.

**Five-way comparison:** base Llama vs fine-tuned Llama vs base GPT-2-XL vs fine-tuned GPT-2-XL vs **fine-tuned LLaDA**.

## Prerequisites

1. **Set runtime to L4 GPU** (Runtime → Change runtime type → L4 GPU). T4 works at MAX_SEQ_LEN=1024 but OOMs at 2048.
2. Training data: run **either** Cell 2a (GitHub) **or** Cell 2b (local upload). Not both.

## Cell Map

| Cell | What it does | Notes |
|------|-------------|-------|
| 0 | This intro | — |
| 1 | Install dependencies | Pins transformers==4.38.2 (LLaDA requirement) |
| 2a | Pull training data from GitHub | SKIP if using 2b |
| 2b | Upload training data locally | SKIP if using 2a |
| 3 | Config + mount Drive | MAX_SEQ_LEN=2048 (needs L4; use 1024 on T4) |
| 4 | Load base LLaDA (4-bit quantization) | float16 compute |
| 5 | Inspect model layers | Verify LoRA target names |
| 6 | Define inference functions | generate_llada, canary_prompts, generate_samples |
| 6a | Smoke test | **Run first — validates pipeline in <2 min** |
| 6b | Generate canary baselines | Full run — check smoke test time estimate first |
| 6c | Save baselines to Drive | |
| 7 | Apply LoRA adapter | |
| 8 | Load + format training data | |
| 9 | Train (custom masked diffusion loop) | float16 autocast |
| 10 | Canary on fine-tuned | |
| 10b | Save fine-tuned outputs | |
| 11 | Side-by-side comparison | |
| 12 | Save LoRA adapter | |

In [ ]:
# === Cell 1: Install dependencies ===
# LLaDA requires transformers==4.38.2 (pinned by the model repo).
# PEFT must be pinned to 0.9.0 — newer versions import EncoderDecoderCache
# which doesn't exist in transformers<4.39.

!pip install -q transformers==4.38.2 datasets peft==0.9.0 bitsandbytes accelerate

In [ ]:
# === Cell 2a: (Option A) Pull training data from GitHub ===
# SKIP this cell if uploading locally via Cell 2b instead.

import os

REPO_URL = "https://github.com/lowyelling/voice-fine-tuning.git"
BRANCH = "demo"  # Change this as needed
REPO_DIR = "/content/voice-fine-tuning"
DATA_DIR = "/content/data"

if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git checkout {BRANCH} && git pull
    print(f"Pulled latest from {BRANCH}.")
else:
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
    print(f"Cloned repo on branch {BRANCH}.")

# Reuse the Llama JSONL files — same chat format, different training loop
os.makedirs(DATA_DIR, exist_ok=True)
!cp {REPO_DIR}/1_data/jsonl/llama_*.jsonl {DATA_DIR}/

print("\nTraining data ready:")
!ls -la {DATA_DIR}/

In [ ]:
# === Cell 2b: (Option B) Upload training data from local computer ===
# SKIP this cell if you already pulled from GitHub via Cell 2a.
#
# Upload llama_train.jsonl and llama_val.jsonl — LLaDA uses the same data format.

import os
from google.colab import files

DATA_DIR = "/content/data"
os.makedirs(DATA_DIR, exist_ok=True)

print("Upload your llama_train.jsonl and llama_val.jsonl files:")
uploaded = files.upload()

for filename, content in uploaded.items():
    filepath = os.path.join(DATA_DIR, filename)
    with open(filepath, "wb") as f:
        f.write(content)
    print(f"  Saved: {filepath}")

print("\nTraining data ready:")
!ls -la {DATA_DIR}/

In [ ]:
# === Cell 3: Config + mount Google Drive ===

import os
import torch
import torch.nn.functional as F
import numpy as np
from transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig
from google.colab import drive

drive.mount("/content/drive")

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

MODEL_ID = "GSAI-ML/LLaDA-8B-Instruct"
MASK_ID = 126336  # LLaDA's [MASK] token — defined in model config, not tokenizer

# LoRA hyperparameters
LORA_R = 16
LORA_ALPHA = 32

# Training hyperparameters
LR = 2e-4
EPOCHS = 3
BATCH_SIZE = 1
MAX_SEQ_LEN = 2048   # Requires A100 (40GB). L4 OOMs at 2048, T4 needs 1024.
                      # No gradient checkpointing (LLaDA doesn't support it).
EPS = 1e-3           # Minimum masking probability (from LLaDA paper)
GRAD_ACCUM_STEPS = 4 # Effective batch size = BATCH_SIZE * GRAD_ACCUM_STEPS = 4
WARMUP_STEPS = 10

# Diffusion inference hyperparameters
# WARNING: LLaDA inference is SLOW — each sample requires steps_per_block x
# num_blocks full forward passes with no KV cache. With the default config
# below, generating 5 samples x 3 canaries takes ~40 min on a T4.
# Use FAST config for testing, FULL config for final comparison.

# --- FAST config (use this while iterating) ---
GEN_LENGTH = 512
GEN_STEPS = 64
GEN_BLOCK_LENGTH = 128
GEN_TEMPERATURE = 0.8
N_SAMPLES = 2

# --- FULL config (uncomment for final comparison) ---
# GEN_LENGTH = 1024
# GEN_STEPS = 128
# GEN_BLOCK_LENGTH = 128
# GEN_TEMPERATURE = 0.8
# N_SAMPLES = 5

# Paths
DATA_DIR = "/content/data"
DRIVE_BASE = "/content/drive/MyDrive/voice-ft"
CHECKPOINT_DIR = f"{DRIVE_BASE}/checkpoints/llada"

print(f"Model: {MODEL_ID}")
print(f"Mask token ID: {MASK_ID}")
print(f"LoRA rank={LORA_R}, alpha={LORA_ALPHA}")
print(f"LR={LR}, epochs={EPOCHS}, batch={BATCH_SIZE}, grad_accum={GRAD_ACCUM_STEPS}")
print(f"Max training seq len: {MAX_SEQ_LEN}")
print(f"Diffusion inference: {GEN_STEPS} steps, {GEN_LENGTH} tokens, temp={GEN_TEMPERATURE}")
print(f"Samples per canary: {N_SAMPLES}")
print(f"Data dir: {DATA_DIR}")
print(f"Checkpoints: {CHECKPOINT_DIR}")

In [ ]:
# === Cell 4: Load base LLaDA (4-bit quantization) ===
#
# Key differences from loading Llama:
# - AutoModel, not AutoModelForCausalLM (LLaDA isn't causal)
# - trust_remote_code=True (loads custom model class from HF repo)
# - No KV cache (bidirectional attention can't cache)
#
# GOTCHA: LLaDA's custom model code calls .to(device) internally after
# loading. bitsandbytes forbids .to() on quantized models (it already
# placed them). We monkey-patch .to() to be a no-op for quantized models.
#
# NOTE: T4 GPUs lack bfloat16 tensor cores (Turing arch). Using bfloat16
# silently falls back to FP32, roughly halving speed. Use float16 instead.

import transformers.modeling_utils
_original_to = transformers.modeling_utils.PreTrainedModel.to

def _patched_to(self, *args, **kwargs):
    if getattr(self, "quantization_method", None) is not None:
        return self
    return _original_to(self, *args, **kwargs)

transformers.modeling_utils.PreTrainedModel.to = _patched_to

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading LLaDA-8B-Instruct in 4-bit...")
print("(This downloads ~4GB and takes a couple minutes)")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.padding_side = "right"

model = AutoModel.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    trust_remote_code=True,
)

total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")
print(f"Model dtype: {next(model.parameters()).dtype}")
print(f"Vocab size: {model.config.embedding_size}")
print(f"Mask token ID: {MASK_ID}")
print(f"\nTokenizer chat template test:")

test_msgs = [{"role": "user", "content": "Hello"}]
print(tokenizer.apply_chat_template(test_msgs, tokenize=False, add_generation_prompt=True))

In [ ]:
# === Cell 5: Inspect model layers (verify LoRA targets) ===
#
# LLaDA uses block_type="llama" internally, so layer names should follow
# Llama conventions. But since it's a custom model class, we verify.
# Look for: q_proj, k_proj, v_proj, o_proj (attention) and
#           gate_proj, up_proj, down_proj (MLP)

print("Model architecture (top-level modules):")
print(type(model))
print()

# Print all named modules that contain 'proj' — these are our LoRA targets
proj_layers = []
for name, module in model.named_modules():
    if 'proj' in name and hasattr(module, 'weight'):
        proj_layers.append((name, tuple(module.weight.shape)))

# Show first block's layers to identify the naming pattern
print("Projection layers in first transformer block:")
for name, shape in proj_layers:
    if '.0.' in name or 'layers.0' in name:
        print(f"  {name}: {shape}")

# Count total
print(f"\nTotal projection layers across all blocks: {len(proj_layers)}")

# Extract unique layer type suffixes (e.g., 'q_proj', 'v_proj')
suffixes = sorted(set(name.split('.')[-1] for name, _ in proj_layers))
print(f"Layer types found: {suffixes}")
print("\n→ Use these as target_modules in LoRA config below.")

In [ ]:
# === Cell 6: Diffusion inference functions ===
# ---------------------------------------------------------------------------
# This is where LLaDA is most different from autoregressive models.
# Instead of model.generate(), we:
#   1. Start with prompt + GEN_LENGTH [MASK] tokens
#   2. Model predicts all masked positions simultaneously
#   3. Unmask the most confident predictions
#   4. Remask the rest and repeat
#
# Think of it like a painter doing successive passes over a canvas,
# filling in the parts they're most sure about first.
# ---------------------------------------------------------------------------

def add_gumbel_noise(logits, temperature):
    """Add Gumbel noise for stochastic sampling. Uses float64 for quality
    (the LLaDA paper found low-precision Gumbel hurts generation)."""
    if temperature == 0:
        return logits
    logits = logits.to(torch.float64)
    noise = torch.rand_like(logits, dtype=torch.float64)
    # Clamp to avoid log(0)
    noise = noise.clamp(min=1e-20)
    gumbel_noise = (-torch.log(noise)) ** temperature
    return logits.exp() / gumbel_noise


def get_num_transfer_tokens(mask_index, steps):
    """Distribute masked tokens evenly across unmasking steps.
    E.g., 128 masked tokens over 16 steps = 8 tokens unmasked per step."""
    mask_num = mask_index.sum(dim=1, keepdim=True)
    base = mask_num // steps
    remainder = mask_num % steps
    num_transfer = torch.zeros(
        mask_num.size(0), steps, device=mask_index.device, dtype=torch.int64
    ) + base
    for i in range(mask_num.size(0)):
        num_transfer[i, :remainder[i]] += 1
    return num_transfer


@torch.no_grad()
def generate_llada(model, prompt_ids, gen_length=GEN_LENGTH, steps=GEN_STEPS,
                   temperature=GEN_TEMPERATURE, block_length=GEN_BLOCK_LENGTH):
    """Generate text using LLaDA's iterative unmasking.

    Args:
        model: LLaDA model
        prompt_ids: tokenized prompt, shape (1, prompt_len)
        gen_length: number of tokens to generate
        steps: total unmasking steps (more = higher quality)
        temperature: sampling temperature (0 = greedy)
        block_length: semi-autoregressive block size

    Returns:
        Full sequence (prompt + generated), shape (1, prompt_len + gen_length)
    """
    device = prompt_ids.device
    prompt_len = prompt_ids.shape[1]

    # Start with prompt + all [MASK] tokens
    x = torch.full(
        (1, prompt_len + gen_length), MASK_ID, dtype=torch.long, device=device
    )
    x[:, :prompt_len] = prompt_ids.clone()

    # Semi-autoregressive: process generation in blocks
    num_blocks = max(1, gen_length // block_length)
    steps_per_block = max(1, steps // num_blocks)

    for block_idx in range(num_blocks):
        block_start = prompt_len + block_idx * block_length
        block_end = min(prompt_len + (block_idx + 1) * block_length,
                        prompt_len + gen_length)

        # How many tokens in this block are still masked?
        block_mask = (x[:, block_start:block_end] == MASK_ID)
        num_transfer = get_num_transfer_tokens(block_mask, steps_per_block)

        for step in range(steps_per_block):
            mask_index = (x == MASK_ID)
            if not mask_index.any():
                break

            # Forward pass — model sees everything, predicts masked positions
            logits = model(x).logits

            # Sample with Gumbel noise
            logits_noisy = add_gumbel_noise(logits, temperature=temperature)
            x0 = torch.argmax(logits_noisy, dim=-1)  # predicted tokens

            # Confidence = softmax probability of the chosen token
            probs = F.softmax(logits.float(), dim=-1)
            x0_confidence = torch.gather(
                probs, dim=-1, index=x0.unsqueeze(-1)
            ).squeeze(-1)

            # Only consider predictions at masked positions
            x0 = torch.where(mask_index, x0, x)
            confidence = torch.where(mask_index, x0_confidence, -torch.inf)

            # Don't unmask tokens in future blocks
            if block_end < x.shape[1]:
                confidence[:, block_end:] = -torch.inf

            # Unmask the top-k most confident tokens this step
            k = num_transfer[0, step].item()
            if k > 0:
                _, top_indices = torch.topk(confidence[0], k=k)
                x[0, top_indices] = x0[0, top_indices]

    return x


# ---------------------------------------------------------------------------
# Canary prompts — same as Llama/GPT-2 for fair comparison
# ---------------------------------------------------------------------------

canary_prompts = {
    "A": "Write a personal Substack Note/Tweet about class in America, told from the perspective of a Chinese first generation immigrant whose family is lower-middle class.",
    "B": "Write a personal Substack Note/Tweet about Eileen Gu and Alyssa Liu, both winter Olympic gold medalists. Both grew up in the Bay Area, are half-asian and half-white, conceived via anonymous egg donor, and raised by a single parent. Eileen competed for China in skiing and is maximizing her influencer career while studying at Stanford. Meanwhile, Alyssa competed for the United States, took breaks from skating, and is inactive on social media.",
    "C": "Write an essay about Jacques Ellul as a forgotten prophet of propaganda and technological conformity.",
}


def generate_samples(model, tokenizer, prompt, n=N_SAMPLES):
    """Generate n samples using diffusion inference."""
    messages = [{"role": "user", "content": prompt}]
    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    input_ids = tokenizer(
        input_text, return_tensors="pt"
    )["input_ids"].to(model.device)
    prompt_len = input_ids.shape[1]

    samples = []
    for i in range(n):
        output = generate_llada(model, input_ids)
        # Decode only the generated part (after prompt)
        generated = tokenizer.decode(
            output[0, prompt_len:], skip_special_tokens=True
        )
        samples.append(generated)

    return samples


print("Inference functions defined.")
print(f"  generate_llada: iterative unmasking ({GEN_STEPS} steps, {GEN_LENGTH} tokens)")
print(f"  generate_samples: wraps generate_llada with chat template")
print(f"  Canary prompts: A (known/short), B (novel/short), C (known/long)")

In [ ]:
# === Cell 6a: Smoke test — validate inference before full baseline run ===
# Generates 128 tokens in 16 steps (~1 block). Should take under 2 min on T4.
# If this takes >5 min, check: is bnb_4bit_compute_dtype set to float16?
# T4 lacks bfloat16 tensor cores — bfloat16 silently falls back to FP32.

import time

model.eval()

test_msgs = [{"role": "user", "content": "Write a short paragraph about rain."}]
input_text = tokenizer.apply_chat_template(test_msgs, tokenize=False, add_generation_prompt=True)
input_ids = tokenizer(input_text, return_tensors="pt")["input_ids"].to(model.device)

print(f"Prompt: {input_ids.shape[1]} tokens")
print(f"Generating 128 tokens in 16 steps (1 block)...")
start = time.time()

output = generate_llada(model, input_ids, gen_length=128, steps=16, temperature=0.8, block_length=128)

elapsed = time.time() - start
print(f"\nDone in {elapsed:.1f}s ({128 / elapsed:.1f} tokens/sec)")
print(f"\n--- Generated text ---")
print(tokenizer.decode(output[0, input_ids.shape[1]:], skip_special_tokens=True))

# Estimate full baseline time
est_per_sample = elapsed * (GEN_LENGTH / 128) * (GEN_STEPS / 16)
est_total = est_per_sample * N_SAMPLES * len(canary_prompts)
print(f"\n--- Time estimate for full baselines ---")
print(f"  {N_SAMPLES} samples x {len(canary_prompts)} canaries x ~{est_per_sample:.0f}s/sample")
print(f"  Estimated total: {est_total/60:.1f} min")

In [ ]:
# === Cell 6b: Generate BASELINE canary outputs ===

model.eval()
baseline_outputs = {}

for name, prompt in canary_prompts.items():
    print(f"\n{'='*60}")
    print(f"BASELINE — Canary {name}")
    print(f"Prompt: {prompt[:80]}...")
    print(f"{'='*60}")

    samples = generate_samples(model, tokenizer, prompt)
    baseline_outputs[name] = samples

    print(f"\nSample 1 of {N_SAMPLES}:")
    print(samples[0][:500])
    print("..." if len(samples[0]) > 500 else "")

print(f"\nBaseline generation complete. {N_SAMPLES} samples per canary saved.")

In [ ]:
# === Cell 6c: Save baselines to Drive ===
import json, os

baseline_path = f"{DRIVE_BASE}/baselines/llada_baselines.json"
os.makedirs(f"{DRIVE_BASE}/baselines", exist_ok=True)

with open(baseline_path, "w") as f:
    json.dump(baseline_outputs, f, indent=2)

print(f"Baselines saved to: {baseline_path}")

In [ ]:
# === Cell 7: Apply LoRA adapter ===
#
# Same concept as Llama LoRA — freeze base weights, add small trainable
# matrices to attention and MLP layers. The fact that LLaDA uses
# bidirectional attention doesn't change how LoRA attaches.
#
# NOTE: LLaDA's layer names differ from Llama:
#   - q_proj, k_proj, v_proj (attention — same as Llama)
#   - ff_proj (MLP — Llama calls this gate_proj)
#   - up_proj (MLP — same as Llama)
#   - No o_proj or down_proj in this architecture
#
# task_type is not set to CAUSAL_LM because LLaDA isn't causal.
#
# GOTCHA: LLaDA's custom model class doesn't support gradient checkpointing
# (raises ValueError). Must pass use_gradient_checkpointing=False.

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "ff_proj", "up_proj"],
    lora_dropout=0.05,
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Should be ~0.3-0.5% trainable. If 0%, something went wrong.

In [ ]:
# === Cell 8: Load + format training data ===
#
# Reuse llama_train.jsonl — same chat format.
# For LLaDA SFT, we need to know where the prompt ends and the response
# begins, because we only mask (and compute loss on) the response.

import json
from torch.utils.data import Dataset, DataLoader


class LLaDADataset(Dataset):
    """Dataset for LLaDA SFT. Each item returns:
    - input_ids: full tokenized conversation (prompt + response)
    - prompt_length: number of tokens in the prompt portion
    """

    def __init__(self, jsonl_path, tokenizer, max_seq_len=MAX_SEQ_LEN):
        self.tokenizer = tokenizer
        self.max_seq_len = max_seq_len
        self.examples = []

        with open(jsonl_path) as f:
            for line in f:
                item = json.loads(line)
                messages = item["messages"]

                # Full conversation: user + assistant
                full_text = tokenizer.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=False
                )

                # Prompt only: user turn + generation prompt marker
                prompt_text = tokenizer.apply_chat_template(
                    messages[:1], tokenize=False, add_generation_prompt=True
                )

                full_ids = tokenizer(
                    full_text, truncation=True, max_length=max_seq_len,
                    return_tensors="pt"
                )["input_ids"].squeeze(0)

                prompt_ids = tokenizer(
                    prompt_text, truncation=True, max_length=max_seq_len,
                    return_tensors="pt"
                )["input_ids"].squeeze(0)

                self.examples.append({
                    "input_ids": full_ids,
                    "prompt_length": prompt_ids.shape[0],
                })

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]


def collate_fn(batch):
    """Pad sequences to the same length within a batch.
    Pad with EOS token (same as pad token in LLaDA)."""
    max_len = max(ex["input_ids"].shape[0] for ex in batch)
    pad_id = tokenizer.pad_token_id or tokenizer.eos_token_id

    input_ids = []
    prompt_lengths = []

    for ex in batch:
        ids = ex["input_ids"]
        pad_len = max_len - ids.shape[0]
        # Pad on the right
        padded = F.pad(ids, (0, pad_len), value=pad_id)
        input_ids.append(padded)
        prompt_lengths.append(ex["prompt_length"])

    return {
        "input_ids": torch.stack(input_ids),
        "prompt_lengths": torch.tensor(prompt_lengths),
    }


# Load datasets
train_path = f"{DATA_DIR}/llama_train.jsonl"
val_path = f"{DATA_DIR}/llama_val.jsonl"

train_dataset = LLaDADataset(train_path, tokenizer)
val_dataset = LLaDADataset(val_path, tokenizer)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn
)

# Sanity checks
print(f"Training examples: {len(train_dataset)}")
print(f"Validation examples: {len(val_dataset)}")

ex = train_dataset[0]
print(f"\nFirst example: {ex['input_ids'].shape[0]} total tokens, "
      f"{ex['prompt_length']} prompt tokens, "
      f"{ex['input_ids'].shape[0] - ex['prompt_length']} response tokens")

# Verify prompt/response split looks right
print(f"\nPrompt (last 100 chars):")
print(tokenizer.decode(ex["input_ids"][:ex["prompt_length"]])[-100:])
print(f"\nResponse (first 200 chars):")
print(tokenizer.decode(ex["input_ids"][ex["prompt_length"]:])[:200])

In [ ]:
# === Cell 9: Train (custom masked diffusion SFT loop) ===
# ---------------------------------------------------------------------------
# This is the core difference from autoregressive fine-tuning.
#
# Autoregressive SFT: predict next token, loss on response tokens.
# LLaDA SFT: randomly mask response tokens, predict them all at once,
#            loss = CE / masking_probability on masked positions.
#
# The 1/p_mask weighting is importance sampling: tokens that were lightly
# masked (small t -> few tokens masked) are rarer training signal, so
# they get upweighted. This ensures the model learns from all masking
# levels equally.
#
# NOTE: No gradient checkpointing — LLaDA's custom model class doesn't
# support it. Training uses more VRAM but still fits on T4 with batch=1.
# ---------------------------------------------------------------------------

import gc
from torch.optim import AdamW
from torch.amp import autocast

gc.collect()
torch.cuda.empty_cache()
print(f"VRAM before training: {torch.cuda.memory_allocated()/1e9:.2f} GB")


def forward_process(input_ids, prompt_lengths, eps=EPS, mask_id=MASK_ID):
    """LLaDA forward diffusion: randomly mask response tokens.

    Returns:
        noisy: input_ids with response tokens randomly replaced by [MASK]
        masked_indices: bool tensor, True where tokens were masked
        p_mask: masking probability per position (for loss weighting)
    """
    b, l = input_ids.shape
    device = input_ids.device

    t = torch.rand(b, device=device)
    p_mask = (1 - eps) * t + eps
    p_mask = p_mask[:, None].expand(b, l)

    mask_draw = torch.rand((b, l), device=device)
    noisy = torch.where(mask_draw < p_mask, mask_id, input_ids)

    positions = torch.arange(l, device=device).expand(b, l)
    prompt_mask = positions < prompt_lengths.unsqueeze(1)
    noisy[prompt_mask] = input_ids[prompt_mask]

    masked_indices = (noisy == mask_id)

    return noisy, masked_indices, p_mask


def compute_loss(model, batch):
    """One training step: mask response tokens, predict, compute weighted loss."""
    input_ids = batch["input_ids"].to(model.device)
    prompt_lengths = batch["prompt_lengths"].to(model.device)

    noisy, masked_indices, p_mask = forward_process(input_ids, prompt_lengths)
    logits = model(input_ids=noisy).logits

    if not masked_indices.any():
        return torch.tensor(0.0, device=model.device, requires_grad=True)

    token_loss = F.cross_entropy(
        logits[masked_indices],
        input_ids[masked_indices],
        reduction='none'
    ) / p_mask[masked_indices]

    b, l = input_ids.shape
    answer_lengths = (l - prompt_lengths).float().unsqueeze(1).expand(b, l)
    loss = torch.sum(token_loss / answer_lengths[masked_indices]) / b

    return loss


# ---------------------------------------------------------------------------
# Training loop
# ---------------------------------------------------------------------------

model.train()

optimizer = AdamW(
    (p for p in model.parameters() if p.requires_grad),
    lr=LR,
    weight_decay=0.01,
)

total_steps = len(train_loader) * EPOCHS // GRAD_ACCUM_STEPS

def get_lr(step):
    if step < WARMUP_STEPS:
        return step / max(1, WARMUP_STEPS)
    progress = (step - WARMUP_STEPS) / max(1, total_steps - WARMUP_STEPS)
    return 0.5 * (1 + np.cos(np.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, get_lr)

print(f"Training LLaDA LoRA")
print(f"  {len(train_dataset)} examples x {EPOCHS} epochs")
print(f"  Batch size: {BATCH_SIZE}, Grad accumulation: {GRAD_ACCUM_STEPS}")
print(f"  Effective batch size: {BATCH_SIZE * GRAD_ACCUM_STEPS}")
print(f"  Total optimizer steps: {total_steps}")
print(f"  Warmup: {WARMUP_STEPS} steps")
print()

train_losses = []
val_losses = []
global_step = 0
best_val_loss = float('inf')

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0
    num_batches = 0

    for batch_idx, batch in enumerate(train_loader):
        with autocast("cuda", dtype=torch.float16):
            loss = compute_loss(model, batch)
            loss = loss / GRAD_ACCUM_STEPS

        loss.backward()

        if (batch_idx + 1) % GRAD_ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(
                model.parameters(), max_norm=1.0
            )
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1

        step_loss = loss.item() * GRAD_ACCUM_STEPS
        epoch_loss += step_loss
        num_batches += 1

        if (batch_idx + 1) % 10 == 0:
            avg = epoch_loss / num_batches
            lr = scheduler.get_last_lr()[0]
            print(f"  Epoch {epoch+1}/{EPOCHS} | Batch {batch_idx+1}/{len(train_loader)} | "
                  f"Loss: {step_loss:.4f} | Avg: {avg:.4f} | LR: {lr:.2e}")

    # Flush remaining gradients if epoch didn't end on an accumulation boundary
    if (batch_idx + 1) % GRAD_ACCUM_STEPS != 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        global_step += 1

    avg_train_loss = epoch_loss / num_batches
    train_losses.append(avg_train_loss)

    # Validation
    model.eval()
    val_loss = 0.0
    val_batches = 0
    with torch.no_grad():
        for batch in val_loader:
            with autocast("cuda", dtype=torch.float16):
                loss = compute_loss(model, batch)
            val_loss += loss.item()
            val_batches += 1

    avg_val_loss = val_loss / max(1, val_batches)
    val_losses.append(avg_val_loss)

    print(f"\n  Epoch {epoch+1}/{EPOCHS} complete: "
          f"train_loss={avg_train_loss:.4f}, val_loss={avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        os.makedirs(CHECKPOINT_DIR, exist_ok=True)
        model.save_pretrained(f"{CHECKPOINT_DIR}/best")
        print(f"  -> New best model saved (val_loss={best_val_loss:.4f})")
    print()

print("\nTraining complete!")
print(f"Final: train_loss={train_losses[-1]:.4f}, val_loss={val_losses[-1]:.4f}")
print(f"Best val_loss: {best_val_loss:.4f}")

In [ ]:
# === Cell 10: Generate FINE-TUNED canary outputs ===

model.eval()
finetuned_outputs = {}

for name, prompt in canary_prompts.items():
    print(f"\n{'='*60}")
    print(f"FINE-TUNED — Canary {name}")
    print(f"Prompt: {prompt[:80]}...")
    print(f"{'='*60}")

    samples = generate_samples(model, tokenizer, prompt)
    finetuned_outputs[name] = samples

    print(f"\nSample 1 of {N_SAMPLES}:")
    print(samples[0][:500])
    print("..." if len(samples[0]) > 500 else "")

print(f"\nFine-tuned generation complete. {N_SAMPLES} samples per canary saved.")

In [ ]:
# === Cell 10b: Save fine-tuned outputs to Drive ===
import json

finetuned_path = f"{DRIVE_BASE}/baselines/llada_finetuned.json"

with open(finetuned_path, "w") as f:
    json.dump(finetuned_outputs, f, indent=2)

print(f"Fine-tuned outputs saved to: {finetuned_path}")

In [ ]:
# === Cell 11: Side-by-side comparison ===

for name, prompt in canary_prompts.items():
    print(f"\n{'#'*70}")
    print(f"  CANARY {name}")
    print(f"  Prompt: {prompt}")
    print(f"{'#'*70}")

    print(f"\n{'─'*35} BASELINE {'─'*35}")
    print(baseline_outputs[name][0])

    print(f"\n{'─'*33} FINE-TUNED {'─'*33}")
    print(finetuned_outputs[name][0])

    print()

print("\n" + "="*70)
print("What to look for:")
print("  - Does the fine-tuned version sound more like you?")
print("  - Is Canary B (novel topic) closer to your voice, or only A (known topic)?")
print("  - If only A improved -> the model memorized content, not voice.")
print("  - If A and B improved but C (long) didn't -> learned sentence voice, not structure.")
print("")
print("LLaDA-specific questions:")
print("  - Does the diffusion generation feel different from Llama's autoregressive output?")
print("  - Is there more internal coherence (since LLaDA sees the whole sequence)?")
print("  - Does the iterative refinement produce better essay-level structure?")
print("="*70)

In [ ]:
# === Cell 12: Save LoRA adapter to Google Drive ===

adapter_path = f"{DRIVE_BASE}/adapters/llada-voice-v1"
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

print(f"LoRA adapter saved to: {adapter_path}")
print(f"\nAdapter size:")
!du -sh {adapter_path}

print(f"\nTo reload this adapter later:")
print(f"""
from peft import PeftModel
from transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModel.from_pretrained(
    "{MODEL_ID}",
    quantization_config=bnb_config,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto",
)

model = PeftModel.from_pretrained(base_model, "{adapter_path}")
tokenizer = AutoTokenizer.from_pretrained("{adapter_path}")
""")